# Система обнаружения стеганографически скрытой информации в цифровых и мультимедийных файлах с помощью нейросетей


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip -q install --upgrade pip

!pip -q install \
  torch torchvision \
  pillow \
  albumentations \
  opencv-python-headless \
  tqdm \
  pandas \
  scikit-learn \
  matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.0 MB/s eta 0:00:00


In [ ]:
# Импорт библиотек
import re
import json
import math
import random
import hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score


In [ ]:
# Загрузка датасета
CANDIDATE_ROOTS = [Path("/content/drive/MyDrive/Colab Notebooks/FQW/Stego-Images-Dataset_archive"), Path("/content/drive/MyDrive/Colab Notebooks/FQW/Stego-Images-Dataset_archive")]
DRIVE_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), None)

if DRIVE_ROOT is None:
    raise RuntimeError("Не найден корень")

print("DRIVE_ROOT =", DRIVE_ROOT)
print("Примеры файлов/папок в корне (первые 10):")
print(sorted([x.name for x in DRIVE_ROOT.iterdir()])[:10])

DRIVE_ROOT = /content/drive/MyDrive/Colab Notebooks/FQW/Stego-Images-Dataset_archive
Примеры файлов/папок в корне (первые 10):
['test', 'train', 'val']


In [ ]:
# DATASET INDEXING

ALLOWED_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

SPLIT_LAYOUT = {
    "train": {"clean": 0, "stego": 1},
    "val":   {"clean": 0, "stego": 1},
    "test":  {"clean": 0, "stego": 1, "stego_b64": 1, "stego_zip": 1},
}

def resolve_split_dir(root: Path, split: str):
    d = root / split
    while (d / split).exists():
        d = d / split
    return d

def list_images(folder: Path):
    return sorted([
        p for p in folder.rglob("*")
        if p.suffix.lower() in ALLOWED_EXTS
    ])

rows = []

for split, subclasses in SPLIT_LAYOUT.items():
    split_dir = resolve_split_dir(DRIVE_ROOT, split)

    for subclass, label in subclasses.items():
        class_dir = split_dir / subclass
        if not class_dir.exists():
            continue

        imgs = list_images(class_dir)

        for p in imgs:
            rows.append({
                "split": split,
                "label": label,
                "path": str(p)
            })

index_df = pd.DataFrame(rows)

print("Всего файлов:", len(index_df))
index_df.head()

Всего файлов: 44000


,split,label,path
0,train,0,/content/drive/MyDrive/Colab Notebooks/FQW/Ste...
1,train,0,/content/drive/MyDrive/Colab Notebooks/FQW/Ste...
2,train,0,/content/drive/MyDrive/Colab Notebooks/FQW/Ste...
3,train,0,/content/drive/MyDrive/Colab Notebooks/FQW/Ste...
4,train,0,/content/drive/MyDrive/Colab Notebooks/FQW/Ste...


In [ ]:
index_df.groupby(["split", "label"]).size()

split  label
test   0         2000
       1        18000
train  0         4000
       1        12000
val    0         2000
       1         6000
dtype: int64

In [ ]:
train_df = index_df[index_df.split == "train"].reset_index(drop=True)
val_df   = index_df[index_df.split == "val"].reset_index(drop=True)
test_df  = index_df[index_df.split == "test"].reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df))

16000 8000 20000


In [ ]:
# FUNCTION: reduce_split (balanced sampling)

SEED = 42

def reduce_split(df, samples_per_class, seed=SEED):
    clean_df = df[df.label == 0]
    stego_df = df[df.label == 1]

    clean_sample = clean_df.sample(
        n=min(samples_per_class, len(clean_df)),
        random_state=seed
    )

    stego_sample = stego_df.sample(
        n=min(samples_per_class, len(stego_df)),
        random_state=seed
    )

    reduced = pd.concat([clean_sample, stego_sample])
    return reduced.sample(frac=1, random_state=seed).reset_index(drop=True)

In [ ]:
# REDUCE ONLY TRAIN TO 1000

train_df = reduce_split(train_df, 500)

print("Train:", len(train_df))
print(train_df.label.value_counts())

Train: 1000
label
1    500
0    500
Name: count, dtype: int64


In [ ]:
IMG_SIZE = 256

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])

In [ ]:
class StegoDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(image=img)["image"]

        label = torch.tensor(row["label"], dtype=torch.float32)
        return img, label

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    StegoDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    StegoDataset(val_df, val_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Device:", device)

Device: cpu


In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for imgs, labels in tqdm(loader):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs).squeeze(1)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs).squeeze(1)
            loss = criterion(outputs, labels)

            probs = torch.sigmoid(outputs)

            total_loss += loss.item()
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds)

    preds_bin = (np.array(all_preds) > 0.5).astype(int)
    acc = (preds_bin == np.array(all_labels)).mean()

    return total_loss / len(loader), acc, auc

In [ ]:
EPOCHS = 5
best_val_auc = 0
best_model_state = None

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader)
    val_loss, val_acc, val_auc = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Acc:    {val_acc:.4f}")
    print(f"Val AUC:    {val_auc:.4f}")

    # Сохраняем лучшую модель по AUC
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model_state = model.state_dict()

  0%|          | 0/32 [00:00<?, ?it/s]


Epoch 1/5
Train Loss: 0.7003
Val Loss:   0.6708
Val Acc:    0.6590
Val AUC:    0.5021


  0%|          | 0/32 [00:00<?, ?it/s]


Epoch 2/5
Train Loss: 0.6940
Val Loss:   0.7133
Val Acc:    0.4066
Val AUC:    0.5052


  0%|          | 0/32 [00:00<?, ?it/s]

...

  0%|          | 0/32 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dca4c5067a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dca4c5067a0>
Exception ignored in: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7dca4c5067a0>    
self._shutdown_workers(


Epoch 4/5
Train Loss: 0.6905
Val Loss:   0.6904
Val Acc:    0.5254
Val AUC:    0.5064


  0%|          | 0/32 [00:00<?, ?it/s]


Epoch 5/5
Train Loss: 0.6916
Val Loss:   0.6789
Val Acc:    0.5824
Val AUC:    0.5059


In [ ]:
# EVALUATION FUNCTION

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs).squeeze(1)
            loss = criterion(logits, labels)

            total_loss += loss.item() * imgs.size(0)

            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(probs)
            all_targets.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    preds_binary = (np.array(all_preds) >= 0.5).astype(int)
    acc = accuracy_score(all_targets, preds_binary)

    try:
        auc = roc_auc_score(all_targets, all_preds)
    except ValueError:
        auc = float("nan")

    return avg_loss, acc, auc

In [ ]:
# RECREATE TEST LOADER

if 'test_df' not in globals():
    raise RuntimeError("test_df не определён. Выполните ячейку разбиения датасета.")

test_loader = DataLoader(
    StegoDataset(test_df, val_transform),   # используем val_transform
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("test_loader recreated.")

test_loader recreated.


In [ ]:
print("model exists:", "model" in globals())
print("criterion exists:", "criterion" in globals())
print("test_loader exists:", "test_loader" in globals())
print("evaluate exists:", "evaluate" in globals())

model exists: True
criterion exists: True
test_loader exists: True
evaluate exists: True


In [ ]:
test_loss, test_acc, test_auc = evaluate(model, test_loader)

print("\nFINAL TEST RESULTS")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_acc:.4f}")
print(f"ROC-AUC: {test_auc:.4f}")


FINAL TEST RESULTS
Loss: 0.7216
Accuracy: 0.1012
ROC-AUC: 0.5003
